# Golden-case regression curation

Runs the pipeline once on a page, lets you browse candidate clusters/segmentations
at three stages that drop/keep bbox'd items -- **vector classification**, **FAST
detection**, **word-splitting (Radon)** -- and capture one hand-verified positive
and one negative case per stage into `outputs/regression_cases/cases.json`. The
final **Replay** section re-runs the pipeline against every stored case (from any
prior curation session too) and reports PASS/FAIL -- the same cell to re-run later
as a regression check.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "golden_case_curation.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rastervec.logging_setup import configure_logging
from rastervec.pipelines.current import run_pipeline
from rastervec.paths import output_dir
from rastervec.renderer.notebook import show_row
from rastervec.Evaluation.Evaluate.golden_schema import (
    GoldenCaseBank,
    load_cases,
    save_cases,
    upsert_case,
)
from rastervec.Evaluation.Evaluate.golden_regression import (
    capture_case,
    format_regression_report,
    list_classification_candidates,
    list_fast_candidates,
    list_word_split_candidates,
    run_regression,
)

configure_logging()

CASES_PATH = str(output_dir("regression_cases") / "cases.json")

## Parameters

In [ ]:
PDF_PATH = next(iter(sorted((PROJECT_ROOT / "references").glob("*.pdf"))), None)
PAGE_INDEX = 3

assert PDF_PATH is not None, "no PDF under references/ -- set PDF_PATH by hand"
print("PDF:", PDF_PATH, "| page", PAGE_INDEX)

## Run the pipeline

In [ ]:
res = run_pipeline(str(PDF_PATH), PAGE_INDEX, enable_fast=True, verbose=True)

try:
    bank = load_cases(CASES_PATH)
except FileNotFoundError:
    bank = GoldenCaseBank()
print(f"loaded {len(bank.cases)} existing case(s) from {CASES_PATH}")

## 1. Vector Classification

Positive candidates are drawn from the final surviving ("kept") text-candidate
clusters; negative candidates from every `role="dropped"` category across the
12-step classification chain, across every (layer, color) bucket.

In [ ]:
classify_pos, classify_neg = list_classification_candidates(res, n=3)

print("positive (kept) candidates:")
for i, c in enumerate(classify_pos):
    print(f"  [{i}] {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
show_row([c.image for c in classify_pos], [f"pos[{i}]" for i in range(len(classify_pos))])

print("negative (dropped) candidates:")
for i, c in enumerate(classify_neg):
    print(f"  [{i}] {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
show_row([c.image for c in classify_neg], [f"neg[{i}]" for i in range(len(classify_neg))])

In [ ]:
# After reading the printed candidates above, set the index of your pick.
CLASSIFY_POS_INDEX = 0
CLASSIFY_NEG_INDEX = 0

In [ ]:
bank = upsert_case(bank, capture_case(
    classify_pos[CLASSIFY_POS_INDEX], stage="classification", label="positive",
    pdf_path=str(PDF_PATH), page_index=PAGE_INDEX,
))
bank = upsert_case(bank, capture_case(
    classify_neg[CLASSIFY_NEG_INDEX], stage="classification", label="negative",
    pdf_path=str(PDF_PATH), page_index=PAGE_INDEX,
))
save_cases(bank, CASES_PATH)
print(f"saved {len(bank.cases)} case(s) to {CASES_PATH}")

## 2. FAST Detection

Positive candidates passed FAST's text-probability threshold; negative candidates
were dropped by it.

In [ ]:
fast_pos, fast_neg = list_fast_candidates(res, n=3)

print("positive (passed) candidates:")
for i, c in enumerate(fast_pos):
    print(f"  [{i}] {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
show_row([c.image for c in fast_pos], [f"pos[{i}]" for i in range(len(fast_pos))])

print("negative (dropped) candidates:")
for i, c in enumerate(fast_neg):
    print(f"  [{i}] {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
show_row([c.image for c in fast_neg], [f"neg[{i}]" for i in range(len(fast_neg))])

In [ ]:
FAST_POS_INDEX = 0
FAST_NEG_INDEX = 0

In [ ]:
bank = upsert_case(bank, capture_case(
    fast_pos[FAST_POS_INDEX], stage="fast", label="positive",
    pdf_path=str(PDF_PATH), page_index=PAGE_INDEX,
))
bank = upsert_case(bank, capture_case(
    fast_neg[FAST_NEG_INDEX], stage="fast", label="negative",
    pdf_path=str(PDF_PATH), page_index=PAGE_INDEX,
))
save_cases(bank, CASES_PATH)
print(f"saved {len(bank.cases)} case(s) to {CASES_PATH}")

## 3. Word Splitting (Radon)

Word-splitting has no structural pass/drop signal -- both picks below come from
one flat pool of segmented clusters (their word boxes drawn in red), and
"positive"/"negative" is purely your own visual judgement of whether the split
looks right.

In [ ]:
word_split_cands = list_word_split_candidates(res, n=6)

print("word-split candidates:")
for i, c in enumerate(word_split_cands):
    print(f"  [{i}] {c.note}  bbox={tuple(round(v, 1) for v in c.bbox)}")
show_row([c.image for c in word_split_cands], [f"[{i}]" for i in range(len(word_split_cands))])

In [ ]:
# Both indices come from the same flat pool above -- see the markdown note.
WORD_SPLIT_POS_INDEX = 0
WORD_SPLIT_NEG_INDEX = 1

In [ ]:
bank = upsert_case(bank, capture_case(
    word_split_cands[WORD_SPLIT_POS_INDEX], stage="word_split", label="positive",
    pdf_path=str(PDF_PATH), page_index=PAGE_INDEX,
))
bank = upsert_case(bank, capture_case(
    word_split_cands[WORD_SPLIT_NEG_INDEX], stage="word_split", label="negative",
    pdf_path=str(PDF_PATH), page_index=PAGE_INDEX,
))
save_cases(bank, CASES_PATH)
print(f"saved {len(bank.cases)} case(s) to {CASES_PATH}")

## Replay

Re-runs the pipeline against every case in the bank (from any prior curation
session too -- not just this one) and reports PASS/FAIL. This is the cell a
contributor re-runs later to check for regressions.

In [ ]:
bank = load_cases(CASES_PATH)
results = run_regression(bank)
print(format_regression_report(results))